# Seeing Machines, Multimodal Archive Companion

A retrieval-augmented pipeline over a personal image corpus, at three levels: **L1 The Finder** (visual search), **L2 The Companion** (caption-grounded RAG) and **L3 The Critic** (hybrid retrieval + multimodal answering), plus a Unified Interface.

---

## Quick start for reviewers (no Google Drive needed)

This notebook runs entirely from the Colab session disk, Google Drive is optional. All artefacts (optimised images, embeddings and captions) are already generated and shipped with the project, so you do not need a GPU and you do not need to re-caption anything.

**1. Upload the project folder.** In the Colab file panel (left sidebar), upload the folder `seeing_machines_project` into `/content/` (or upload the provided ZIP and unzip it there). The folder already has the required structure:

```
/content/seeing_machines_project/
├── input_images/              (originals)
├── optimized_images/          (512px JPEGs)
├── image_paths.pkl
├── image_embeddings.pkl
├── corpus_descriptions.json   (Gemma captions, no need to regenerate)
└── description_embeddings.pkl
```

The folder name and location matter: it must be exactly `/content/seeing_machines_project/`, because the cached file lists reference that path.

**2. Run the setup cells in order:** Cell 1 (install), skip Cell 2 (Drive) if you don't want to mount it, then Cell 3, then Cell 4. Cell 4 scans the folder and reports "all artefacts cached" with a short plan.

**3. Leave every switch in Cell 5 set to `False`** (this reuses the cached files instead of regenerating them).

**4. Load and launch:**
- L1 (no GPU): run Cell 7, 8, 9, then Cell 10 to open the visual-search interface.
- L2 / L3 (need a GPU for answer generation): attach a T4 GPU (Runtime → Change runtime type → T4), then run Cell 11, 12, 13, 14, 15 for L2, Cell 17, 18 for L3, and Cell 20 for the unified app. With the cached captions, Cell 12 just loads them, it does not re-run Gemma.

Do not use "Run all". Cells 10, 15, 18 and 20 launch interactive Gradio servers that keep running until you stop them, so a single "Run all" would halt at the first interface. Run the cells in order, launching each interface when you reach it.

Optional, with Google Drive: run Cell 2 to mount Drive. If the artefacts live in your Drive instead of the uploaded folder, Cell 4 copies them into the runtime automatically, and anything you generate is mirrored back to Drive.

---

## How the notebook is organised (by level)

You run the shared setup once, then each level's data cells sit directly before that level's Gradio app:

- **Setup (shared):** install, Drive, configuration, readiness check, regeneration control.
- **Shared retrieval core:** image optimisation, CLIP model, image embeddings, L1 search function.
- **L1, The Finder:** its Gradio app.
- **L2, The Companion:** load Gemma, caption the images, embed the captions, L2 functions, then its Gradio app, plus the L1-vs-L2 route comparison.
- **L3, The Critic:** hybrid + multimodal functions, its Gradio app, plus the precision@k evaluation.
- **Unified Interface:** all three levels in tabs.

## Caching model

Every artefact lives in two mirrored folders: `RUNTIME_DIR` (the session disk, wiped when the runtime is deleted) and, if Drive is mounted, `DRIVE_DIR` (permanent). Each heavy cell reuses the runtime copy, else copies it from Drive, else generates it and saves it. What gets rebuilt is chosen in one place: Cell 5, Regeneration control.

## Cell index

| # | Section | Level | GPU |
|---|---------|-------|-----|
| 1 | Install dependencies | setup | no |
| 2 | Mount Google Drive (optional) | setup | no |
| 3 | Project configuration & cache helpers | setup | no |
| 4 | Readiness check | setup | no |
| 5 | Regeneration control | setup | no |
| 6 | Image optimisation | shared | no |
| 7 | Load the CLIP model | shared | no |
| 8 | Image embeddings | shared | no |
| 9 | L1 search function | L1 | no |
| 10 | L1, The Finder (Gradio) | L1 | no |
| 11 | Load Gemma 3 VLM (4-bit) | L2 | yes |
| 12 | Image descriptions / captioning | L2 | yes |
| 13 | Description embeddings | L2 | no |
| 14 | L2 retrieval & RAG functions | L2 | no |
| 15 | L2, The Companion (Gradio) | L2 | yes |
| 16 | Route comparison, L1 vs L2 | L2 | no |
| 17 | L3 hybrid & multimodal functions | L3 | no |
| 18 | L3, The Critic (Gradio) | L3 | yes |
| 19 | Evaluation, precision@k | L3 | no |
| 20 | Fusion weight sweep (alpha) | L3 | no |
| 21 | Unified Interface | all | yes |
| 22 | Experiment: CLIP vs SigLIP2 | L3 | yes |
| 23 | Comparison chart | L3 | no |
| 24 | Optional utility: gold-set builder | L3 | no |

The GPU is only needed for the Gemma cells (11 to 12) and L2/L3 answer generation. Everything else, including all of L1, runs on a CPU runtime.

## Cell 1. Install dependencies

Installs only the packages Colab does not already ship with: `gradio`, `bitsandbytes` (4-bit loading), `pillow-heif` (HEIC/HEIF), `rawpy` (RAW .dng), and upgrades `transformers` only if it is older than 4.50 (Gemma 3 support).

`torch`, `Pillow`, `accelerate` and `tqdm` are pre-installed in Colab and are **not** touched: upgrading `torch` pulls in a CUDA stack that conflicts with other pre-installed Colab libraries.


In [1]:
# Install only what is missing. torch / torchvision are deliberately NOT
# installed or upgraded here (Colab already provides a matching CUDA build).
# transformers>=4.50 also covers SigLIP2 (Cell 22); matplotlib for the chart
# (Cell 23) ships with Colab, so it is not installed here.
%pip install -q gradio bitsandbytes pillow-heif rawpy "transformers>=4.50"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 62.4 MB/s eta 0:00:00


## Cell 2. Mount Google Drive (optional, strongly recommended)

Mounts your Drive at `/content/drive`, the permanent home of all artefacts so they survive Colab disconnects. If you skip this cell (or run locally) the notebook still works using only the runtime folder, but generated files are lost when the runtime is deleted. Cell 4 will offer to mount Drive if it detects missing files.


In [2]:
# Try to mount Google Drive; fall back gracefully to runtime-only mode.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_AVAILABLE = True
    print('OK  Google Drive mounted at /content/drive')
except Exception as e:
    DRIVE_AVAILABLE = False
    print(f'WARNING  Google Drive not available ({e}).')
    print('Running in RUNTIME-ONLY mode: generated files are NOT persisted.')
    print('Download them before deleting the runtime.')


WARNING  Google Drive not available (Error: credential propagation was unsuccessful).
Running in RUNTIME-ONLY mode: generated files are NOT persisted.
Download them before deleting the runtime.


## Cell 3. Project configuration & cache helpers

Defines the mirrored folders and artefact names, plus the helpers used everywhere: `save_artifact()` (write to runtime + Drive, replacing older copies), `ensure_runtime_copy()` (pull from Drive when needed), `try_mount_drive()`, `should_regenerate()` (reads the central `REGEN` config from Cell 5), and `preflight()` (checks that a Gradio cell's requirements are loaded and lists the cells to run otherwise).

This cell must always be run.


In [3]:
import os, shutil

# 1. Folder layout -----------------------------------------------------------
RUNTIME_DIR = '/content/seeing_machines_project'                # session disk
DRIVE_ROOT  = '/content/drive/MyDrive/seeing_machines_project'  # Drive mirror

if 'DRIVE_AVAILABLE' not in globals():
    DRIVE_AVAILABLE = False
DRIVE_DIR = DRIVE_ROOT if DRIVE_AVAILABLE else None

def _subdirs():
    """(Re)compute every sub-folder path from the current DRIVE_DIR value."""
    global RUNTIME_INPUT_DIR, RUNTIME_OPTIMIZED_DIR, DRIVE_INPUT_DIR, DRIVE_OPTIMIZED_DIR
    RUNTIME_INPUT_DIR     = os.path.join(RUNTIME_DIR, 'input_images')
    RUNTIME_OPTIMIZED_DIR = os.path.join(RUNTIME_DIR, 'optimized_images')
    DRIVE_INPUT_DIR       = os.path.join(DRIVE_DIR, 'input_images')     if DRIVE_DIR else None
    DRIVE_OPTIMIZED_DIR   = os.path.join(DRIVE_DIR, 'optimized_images') if DRIVE_DIR else None
    for d in [RUNTIME_DIR, RUNTIME_INPUT_DIR, RUNTIME_OPTIMIZED_DIR,
              DRIVE_DIR, DRIVE_INPUT_DIR, DRIVE_OPTIMIZED_DIR]:
        if d:
            os.makedirs(d, exist_ok=True)
_subdirs()

FILE_NAMES = {
    'image_paths'           : 'image_paths.pkl',
    'image_embeddings'      : 'image_embeddings.pkl',
    'descriptions'          : 'corpus_descriptions.json',
    'description_embeddings': 'description_embeddings.pkl',
}
IMAGE_EXTS = ('.png', '.jpg', '.jpeg', '.heic', '.heif', '.webp', '.bmp', '.dng')

def runtime_path(key): return os.path.join(RUNTIME_DIR, FILE_NAMES[key])
def drive_path(key):   return os.path.join(DRIVE_DIR, FILE_NAMES[key]) if DRIVE_DIR else None

# 2. Cache helpers -----------------------------------------------------------
def ensure_runtime_copy(key):
    """Ensure an artefact is in the runtime folder, copying from Drive if needed."""
    rp = runtime_path(key)
    if os.path.exists(rp):
        return rp
    dp = drive_path(key)
    if dp and os.path.exists(dp):
        shutil.copy2(dp, rp)
        print(f'COPIED  {FILE_NAMES[key]}  Drive -> runtime.')
        return rp
    return None

def save_artifact(key):
    """Mirror a freshly written runtime artefact to Drive (replacing the old
    copy), or warn that only a temporary runtime copy exists."""
    rp, dp = runtime_path(key), drive_path(key)
    if not os.path.exists(rp):
        print(f'ERROR  {FILE_NAMES[key]} not found in the runtime folder.')
        return
    if dp:
        shutil.copy2(rp, dp)
        print(f'SAVED  {FILE_NAMES[key]} -> runtime AND Google Drive (replaced).')
    else:
        print(f'SAVED  {FILE_NAMES[key]} -> runtime ONLY (Drive not mounted).')
        print('       Download it before deleting the runtime, or mount Drive')
        print('       (Cell 2) and re-run this cell to mirror it.')

def try_mount_drive():
    """Mount Drive on demand (used by the readiness check)."""
    global DRIVE_AVAILABLE, DRIVE_DIR
    if DRIVE_AVAILABLE:
        return True
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_AVAILABLE = True
        DRIVE_DIR = DRIVE_ROOT
        _subdirs()
        print('OK  Google Drive mounted.')
        return True
    except Exception as e:
        print(f'WARNING  Could not mount Drive ({e}). Continuing runtime-only.')
        return False

# 3. Regeneration control (reads the REGEN dict defined in Cell 5) ------------
def should_regenerate(key):
    """Return True if the artefact `key` must be (re)built.
      * No cached copy anywhere -> generate regardless of the flag.
      * Otherwise -> follow REGEN[key] from Cell 5."""
    cached = os.path.exists(runtime_path(key)) or (drive_path(key) and os.path.exists(drive_path(key)))
    flag = globals().get('REGEN', {}).get(key, False)
    if not cached:
        print(f'No cached {FILE_NAMES[key]} found -> generating it now.')
        return True
    if flag:
        print(f'REGEN["{key}"] = True -> rebuilding {FILE_NAMES[key]} and replacing '
              'the runtime' + (' and Drive copies.' if DRIVE_DIR else ' copy.'))
        return True
    print(f'Using the cached {FILE_NAMES[key]} '
          f'(set REGEN["{key}"] = True in Cell 5 to rebuild it).')
    return False

# 4. Preflight guard for the Gradio cells ------------------------------------
REQUIREMENTS = {
    'clip_model'            : ('CLIP model (`clip_model`)',                       'Cell 7'),
    'image_embeddings'      : ('image embeddings (`image_embeddings`)',           'Cell 8'),
    'images'                : ('optimised image list (`images`)',                 'Cell 6 (or Cell 8, which loads them from cache)'),
    'search_images'         : ('L1 search function (`search_images`)',            'Cell 9'),
    'vlm_model'             : ('Gemma 3 VLM (`vlm_model`)',                       'Cell 11'),
    'description_texts'     : ('image descriptions (`description_texts`)',        'Cell 12'),
    'description_embeddings': ('description embeddings (`description_embeddings`)','Cell 13'),
    'l2_rag_answer'         : ('L2 RAG function (`l2_rag_answer`)',               'Cell 14'),
    'l3_multimodal_rag'     : ('L3 multimodal function (`l3_multimodal_rag`)',    'Cell 17'),
}

def preflight(names, interface_name=''):
    """Verify required objects exist; otherwise list the exact cells to run."""
    g = globals()
    missing = [n for n in names if n not in g or g[n] is None]
    if not missing:
        print(f'OK  All requirements satisfied{" for " + interface_name if interface_name else ""}.')
        return
    print(f'MISSING requirements{" for " + interface_name if interface_name else ""}:')
    for n in missing:
        desc, cell = REQUIREMENTS[n]
        print(f'  - {desc}  ->  run {cell}')
    raise RuntimeError('Interface not launched. Run the cells listed above, then re-run this cell.')

print(f'Runtime folder : {RUNTIME_DIR}')
print(f'Drive folder   : {DRIVE_DIR if DRIVE_DIR else "(not mounted)"}')
print('Place your original images in either input_images folder; they are mirrored.')


Runtime folder : /content/seeing_machines_project
Drive folder   : (not mounted)
Place your original images in either input_images folder; they are mirrored.


## Cell 4. Readiness check: your guided plan

Scans the runtime (and Drive) for every file the pipeline needs, mounts Drive automatically if something is missing, copies any cached artefacts from Drive into the runtime, then prints a status table and a step-by-step plan: which cells to skip, which to run and in what order, or what to upload.


In [5]:
import os, shutil

def _count_images(folder):
    if not folder or not os.path.isdir(folder):
        return 0
    return len([f for f in os.listdir(folder) if f.lower().endswith(IMAGE_EXTS)])

def _pull_missing_from_drive():
    for key in FILE_NAMES:
        ensure_runtime_copy(key)
    for src_dir, dst_dir, label in [(DRIVE_INPUT_DIR, RUNTIME_INPUT_DIR, 'input'),
                                    (DRIVE_OPTIMIZED_DIR, RUNTIME_OPTIMIZED_DIR, 'optimised')]:
        if not (src_dir and os.path.isdir(src_dir)):
            continue
        copied = 0
        for f in os.listdir(src_dir):
            if f.lower().endswith(IMAGE_EXTS) and not os.path.exists(os.path.join(dst_dir, f)):
                shutil.copy2(os.path.join(src_dir, f), os.path.join(dst_dir, f))
                copied += 1
        if copied:
            print(f'COPIED  {copied} {label} image(s) from Drive to the runtime.')

def readiness_check():
    def scan():
        artefacts = {k: os.path.exists(runtime_path(k)) for k in FILE_NAMES}
        return artefacts, _count_images(RUNTIME_INPUT_DIR), _count_images(RUNTIME_OPTIMIZED_DIR)

    artefacts, n_input, n_opt = scan()
    anything_missing = (not all(artefacts.values())) or (n_input == 0 and n_opt == 0)
    if anything_missing and not DRIVE_AVAILABLE:
        print('Some required files were not found; mounting Drive to look there...')
        try_mount_drive()
    if DRIVE_AVAILABLE:
        _pull_missing_from_drive()
        artefacts, n_input, n_opt = scan()

    print()
    print('STATUS'); print('-' * 62)
    print(f'{"artefact":<34}{"runtime":<10}{"drive":<10}'); print('-' * 62)
    for key, name in FILE_NAMES.items():
        in_rt = artefacts[key]
        in_dr = bool(drive_path(key)) and os.path.exists(drive_path(key))
        print(f'{name:<34}{("yes" if in_rt else "NO"):<10}{("yes" if in_dr else ("NO" if DRIVE_AVAILABLE else "-")):<10}')
    print('-' * 62)
    print(f'{"input_images (originals)":<34}{n_input:<10}{_count_images(DRIVE_INPUT_DIR) if DRIVE_AVAILABLE else "-"}')
    print(f'{"optimized_images (512px JPEG)":<34}{n_opt:<10}{_count_images(DRIVE_OPTIMIZED_DIR) if DRIVE_AVAILABLE else "-"}')
    print('-' * 62)

    print(); print('YOUR PLAN')
    all_cached  = all(artefacts.values())
    have_images = n_input > 0 or n_opt > 0

    if all_cached:
        print('All heavy artefacts are cached. Fastest route:')
        print('  L1 only : Cell 7 (CLIP) -> Cell 8 (loads embeddings) -> Cell 9 -> Cell 10')
        print('  L2      : + Cell 11 (Gemma) -> Cell 12 (loads captions) -> Cell 13 -> Cell 14 -> Cell 15')
        print('  L3      : + Cell 17 -> Cell 18')
        print('  All     : Cell 20 (Unified) after the cells above are loaded')
    elif not have_images:
        print('No images found. Upload 50-150 images to:')
        print(f'  runtime: {RUNTIME_INPUT_DIR}')
        if DRIVE_AVAILABLE:
            print(f'  or Drive (persists): {DRIVE_INPUT_DIR}')
        print('Then re-run this cell.')
    else:
        print('Images found, but some artefacts must be generated. Run in this order:')
        step = 1
        if n_opt == 0 or not artefacts['image_paths']:
            print(f'  {step}. Cell 6  - optimise the images'); step += 1
        print(f'  {step}. Cell 7  - load CLIP'); step += 1
        if not (artefacts['image_embeddings'] and artefacts['image_paths']):
            print(f'  {step}. Cell 8  - generate image embeddings'); step += 1
        else:
            print(f'  {step}. Cell 8  - loads cached image embeddings'); step += 1
        print(f'  {step}. Cell 9  - define the L1 search function'); step += 1
        print(f'  {step}. Cell 10 - L1 Finder (works now; L2/L3 need the steps below)'); step += 1
        print(f'  {step}. Cell 11 - load Gemma 3 (ATTACH THE GPU)'); step += 1
        if not artefacts['descriptions']:
            print(f'  {step}. Cell 12 - generate captions'); step += 1
        else:
            print(f'  {step}. Cell 12 - loads cached captions'); step += 1
        if not artefacts['description_embeddings']:
            print(f'  {step}. Cell 13 - generate caption embeddings'); step += 1
        else:
            print(f'  {step}. Cell 13 - loads cached caption embeddings'); step += 1
        print(f'  {step}. Cell 14 -> 15 (L2), Cell 17 -> 18 (L3), Cell 20 (Unified)')
        if not DRIVE_AVAILABLE:
            print()
            print('WARNING: Drive is NOT mounted. Generated files live only in the temporary')
            print('runtime - download them, or mount Drive (Cell 2) to mirror them.')
    return all_cached

ALL_READY = readiness_check()



STATUS
--------------------------------------------------------------
artefact                          runtime   drive     
--------------------------------------------------------------
image_paths.pkl                   yes       -         
image_embeddings.pkl              yes       -         
corpus_descriptions.json          yes       -         
description_embeddings.pkl        yes       -         
--------------------------------------------------------------
input_images (originals)          0         -
optimized_images (512px JPEG)     56        -
--------------------------------------------------------------

YOUR PLAN
All heavy artefacts are cached. Fastest route:
  L1 only : Cell 7 (CLIP) -> Cell 8 (loads embeddings) -> Cell 9 -> Cell 10
  L2      : + Cell 11 (Gemma) -> Cell 12 (loads captions) -> Cell 13 -> Cell 14 -> Cell 15
  L3      : + Cell 17 -> Cell 18
  All     : Cell 20 (Unified) after the cells above are loaded


## Cell 5. Regeneration control (set your choices once)

The single place to decide what gets rebuilt. Each `REGEN` entry: `False` reuses the cache, `True` rebuilds and **replaces** the copies in the runtime and in Drive. An artefact that exists nowhere is generated regardless. `REGENERATE_ALL = True` forces a full rebuild.

Rebuilding an upstream artefact usually means rebuilding the ones that depend on it: `optimized -> image_embeddings -> descriptions -> description_embeddings`.


In [6]:
# Master switch: True rebuilds EVERYTHING (overrides the individual flags).
REGENERATE_ALL = False

REGEN = {
    'optimized'             : False,   # Cell 6  - resize/convert originals
    'image_embeddings'      : False,   # Cell 8  - CLIP visual embeddings
    'descriptions'          : False,   # Cell 12 - Gemma captions (slow, GPU)
    'description_embeddings': False,   # Cell 13 - CLIP text embeddings of captions
}
if REGENERATE_ALL:
    REGEN = {k: True for k in REGEN}

print('Regeneration plan (True = rebuild + replace, False = reuse cache):')
for _k, _v in REGEN.items():
    print(f'  {_k:<24} {_v}')
if REGENERATE_ALL:
    print('REGENERATE_ALL = True -> every artefact will be rebuilt.')
print('\nReminder: rebuild dependents too '
      '(optimized -> image_embeddings -> descriptions -> description_embeddings).')


Regeneration plan (True = rebuild + replace, False = reuse cache):
  optimized                False
  image_embeddings         False
  descriptions             False
  description_embeddings   False

Reminder: rebuild dependents too (optimized -> image_embeddings -> descriptions -> description_embeddings).


## Cell 6. Image optimisation (smart cache), *shared by all levels*

Resizes every original so its long side is 512 px, converts to RGB, and saves a JPEG in `optimized_images/`. HEIC/HEIF and RAW (.dng) are converted here. Reads `REGEN['optimized']` from Cell 5; on rebuild it deletes the previous optimised set (runtime + Drive) before regenerating.


In [7]:
from PIL import Image
import os, shutil
import rawpy
from pillow_heif import register_heif_opener
register_heif_opener()          # teach PIL to open .heic / .heif files

def _collect_input_images():
    """Unique originals from both input folders (Drive-only ones copied in)."""
    all_files = set()
    if os.path.isdir(RUNTIME_INPUT_DIR):
        all_files.update(f for f in os.listdir(RUNTIME_INPUT_DIR) if f.lower().endswith(IMAGE_EXTS))
    if DRIVE_INPUT_DIR and os.path.isdir(DRIVE_INPUT_DIR):
        for f in os.listdir(DRIVE_INPUT_DIR):
            if f.lower().endswith(IMAGE_EXTS) and f not in all_files:
                shutil.copy2(os.path.join(DRIVE_INPUT_DIR, f), os.path.join(RUNTIME_INPUT_DIR, f))
                all_files.add(f)
    return sorted(os.path.join(RUNTIME_INPUT_DIR, f) for f in all_files)

def _load_optimized_from_disk():
    """Load already-optimised JPEGs directly, without needing the originals.
    Returns (PIL images, paths) in stable sorted order."""
    files = sorted(f for f in os.listdir(RUNTIME_OPTIMIZED_DIR) if f.lower().endswith('.jpg'))
    paths = [os.path.join(RUNTIME_OPTIMIZED_DIR, f) for f in files]
    imgs  = [Image.open(p).convert('RGB') for p in paths]
    return imgs, paths

def open_image_with_raw_support(path):
    """Open .dng via rawpy; PIL for everything else."""
    if path.lower().endswith('.dng'):
        with rawpy.imread(path) as raw:
            return Image.fromarray(raw.postprocess(use_camera_wb=True))
    return Image.open(path)

def _wipe_optimized():
    for folder in [RUNTIME_OPTIMIZED_DIR, DRIVE_OPTIMIZED_DIR]:
        if folder and os.path.isdir(folder):
            for f in os.listdir(folder):
                if f.lower().endswith('.jpg'):
                    os.remove(os.path.join(folder, f))

def optimize_corpus(target_long_side=512, force=False):
    """Resize originals; return (PIL images, optimised paths) in stable order.
    Requires the original images (only called when a rebuild is needed)."""
    raw_paths = _collect_input_images()
    if not raw_paths:
        raise RuntimeError(
            f'No original images found in {RUNTIME_INPUT_DIR}. '
            'To (re)build the optimised set you need the originals. '
            'If you only have the cached optimised_images folder, set '
            'REGEN["optimized"] = False in Cell 5 so this cell loads them directly.')
    optimized_images, optimized_paths = [], []
    for original_path in raw_paths:
        base        = os.path.splitext(os.path.basename(original_path))[0] + '.jpg'
        runtime_out = os.path.join(RUNTIME_OPTIMIZED_DIR, base)
        drive_out   = os.path.join(DRIVE_OPTIMIZED_DIR, base) if DRIVE_OPTIMIZED_DIR else None
        if not force and os.path.exists(runtime_out):
            img = Image.open(runtime_out).convert('RGB')
        else:
            try:
                img = open_image_with_raw_support(original_path).convert('RGB')
                w, h = img.size
                if max(w, h) > target_long_side:
                    scale = target_long_side / max(w, h)
                    img = img.resize((round(w * scale), round(h * scale)), Image.Resampling.LANCZOS)
                img.save(runtime_out, 'JPEG', quality=90)
                if drive_out:
                    shutil.copy2(runtime_out, drive_out)
                print(f'PROCESSED  {base}')
            except Exception as e:
                print(f'ERROR  {original_path}: {e}')
                continue
        optimized_images.append(img)
        optimized_paths.append(runtime_out)
    print(f'\nDONE  {len(optimized_images)} images ready in {RUNTIME_OPTIMIZED_DIR}')
    if DRIVE_OPTIMIZED_DIR:
        print(f'      Mirrored to Google Drive: {DRIVE_OPTIMIZED_DIR}')
    else:
        print('      WARNING: Drive not mounted - optimised images are in the runtime ONLY.')
    return optimized_images, optimized_paths

# ---------------------------------------------------------------------------
# Decide what to do, reading the central REGEN config (Cell 5).
# Priority:
#   1. REGEN['optimized'] = True  -> rebuild from originals (needs input_images).
#   2. Optimised JPEGs already exist -> load them directly (originals NOT needed).
#   3. Otherwise -> optimise from originals.
# ---------------------------------------------------------------------------
_has_opt = os.path.isdir(RUNTIME_OPTIMIZED_DIR) and any(
    f.lower().endswith('.jpg') for f in os.listdir(RUNTIME_OPTIMIZED_DIR))

if REGEN.get('optimized', False):
    # Explicit rebuild requested.
    if _has_opt:
        print('REGEN["optimized"] = True -> replacing the optimised image set.')
        _wipe_optimized()
    images, image_paths = optimize_corpus(force=True)
elif _has_opt:
    # Cache hit: load the optimised JPEGs directly, no originals required.
    print('Using the cached optimised images (loaded directly, originals not needed).')
    print('Set REGEN["optimized"] = True in Cell 5 to rebuild them from the originals.')
    images, image_paths = _load_optimized_from_disk()
    print(f'OK  Loaded {len(images)} optimised images from {RUNTIME_OPTIMIZED_DIR}')
else:
    # Nothing cached: must optimise from originals.
    print('No optimised images yet -> optimising from the originals now.')
    images, image_paths = optimize_corpus(force=False)

Using the cached optimised images (loaded directly, originals not needed).
Set REGEN["optimized"] = True in Cell 5 to rebuild them from the originals.
OK  Loaded 56 optimised images from /content/seeing_machines_project/optimized_images


## Cell 7. Load the CLIP model, *shared by all levels*

Loads `openai/clip-vit-base-patch32`, the joint text–image embedding model used by L1 (visual search), L2 (embedding captions and the query) and L3 (hybrid retrieval). Also defines `clip_image_features` / `clip_text_features`, small wrappers that always return a plain tensor regardless of the installed `transformers` version.


In [8]:
from transformers import CLIPProcessor, CLIPModel
import torch

device  = 'cuda' if torch.cuda.is_available() else 'cpu'
clip_id = 'openai/clip-vit-base-patch32'
clip_model     = CLIPModel.from_pretrained(clip_id).to(device)
clip_processor = CLIPProcessor.from_pretrained(clip_id)

def _as_tensor(out):
    """Some transformers versions return a ModelOutput instead of a tensor.
    Extract the projected feature tensor in every case."""
    if isinstance(out, torch.Tensor):
        return out
    if hasattr(out, 'image_embeds') and out.image_embeds is not None:
        return out.image_embeds
    if hasattr(out, 'text_embeds') and out.text_embeds is not None:
        return out.text_embeds
    if hasattr(out, 'pooler_output') and out.pooler_output is not None:
        return out.pooler_output
    if hasattr(out, 'last_hidden_state'):
        return out.last_hidden_state[:, 0]   # CLS token fallback
    raise TypeError(f'Unexpected CLIP output type: {type(out)}')

def clip_image_features(pil_image):
    """CLIP visual features of one image as a CPU tensor (1, D)."""
    inputs = clip_processor(images=pil_image, return_tensors='pt').to(device)
    with torch.no_grad():
        return _as_tensor(clip_model.get_image_features(**inputs)).cpu()

def clip_text_features(text):
    """CLIP text features of one string as a CPU tensor (1, D)."""
    inputs = clip_processor(text=[text], return_tensors='pt',
                            padding=True, truncation=True).to(device)  # CLIP max = 77 tokens
    with torch.no_grad():
        return _as_tensor(clip_model.get_text_features(**inputs)).cpu()

print(f'OK  CLIP loaded on {device}.')


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

OK  CLIP loaded on cuda.


## Cell 8. Image embeddings (smart cache), *shared by all levels*

Builds or loads the CLIP visual embeddings (one vector per image). When loading from cache it also restores the `images` list from disk, so Cell 6 need not be re-run on a fresh runtime. Reads `REGEN['image_embeddings']` from Cell 5.


In [9]:
import pickle, torch
from PIL import Image

def _ensure_2d(t):
    """Normalise any cached embedding object to a single (N, D) tensor."""
    if isinstance(t, list):
        t = torch.cat([x if x.ndim == 2 else x.unsqueeze(0) for x in t])
    if t.ndim == 1:
        t = t.unsqueeze(0)
    elif t.ndim > 2:
        t = t.reshape(t.shape[0], -1)
    return t

def _load_pickle(p):
    with open(p, 'rb') as f: return pickle.load(f)
def _save_pickle(p, o):
    with open(p, 'wb') as f: pickle.dump(o, f)

def build_image_embeddings(image_list):
    """Encode every image with CLIP's vision tower -> tensor (N, D)."""
    return torch.cat([clip_image_features(img) for img in image_list])

if should_regenerate('image_embeddings'):
    if 'images' not in globals() or not images:
        raise RuntimeError('No optimised images in memory. Run Cell 6 first (see Cell 4).')
    print(f'GENERATING embeddings for {len(images)} images...')
    image_embeddings = _ensure_2d(build_image_embeddings(images))
    _save_pickle(runtime_path('image_embeddings'), image_embeddings)
    _save_pickle(runtime_path('image_paths'),      image_paths)
    save_artifact('image_embeddings')
    save_artifact('image_paths')
    print(f'OK  Image embeddings ready (shape {tuple(image_embeddings.shape)}).')
else:
    _emb   = ensure_runtime_copy('image_embeddings')
    _paths = ensure_runtime_copy('image_paths')
    image_embeddings = _ensure_2d(_load_pickle(_emb))
    image_paths      = _load_pickle(_paths)
    images           = [Image.open(p).convert('RGB') for p in image_paths]
    print(f'OK  Loaded {image_embeddings.shape[0]} cached image embeddings '
          f'(shape {tuple(image_embeddings.shape)}).')


Using the cached image_embeddings.pkl (set REGEN["image_embeddings"] = True in Cell 5 to rebuild it).
OK  Loaded 56 cached image embeddings (shape (56, 512)).


## Cell 9. L1 search function

Defines the visual-search primitives: `embed_query` (CLIP text embedding of the query), `search_images_idx` (top-k indices + similarities against the image embeddings) and `search_images` (returns image + score labels for the Gradio gallery). Instant to run.


In [10]:
import os
import torch, torch.nn.functional as F

def embed_query(query):
    """CLIP text embedding of a query, shape (1, D)."""
    emb = clip_text_features(query)
    return emb if emb.ndim == 2 else emb.unsqueeze(0)

def search_images_idx(query, top_k=3):
    """Top-k image indices + full similarity vector (visual route)."""
    q    = embed_query(query)
    sims = F.cosine_similarity(q, image_embeddings)
    top  = sims.topk(min(int(top_k), len(image_embeddings)))
    return top.indices.tolist(), sims

def search_images(query, top_k=3):
    """L1 visual search -> (image, label) pairs for the gallery."""
    idx, sims = search_images_idx(query, top_k)
    return [(images[i], f'{os.path.basename(image_paths[i])}  |  sim={sims[i]:.3f}')
            for i in idx]

print('OK  L1 search function defined.')


OK  L1 search function defined.


## Cell 10. L1: The Finder (Gradio)

Pure visual search: the text query is embedded by CLIP and matched against the image embeddings. Each result shows its file name and cosine similarity. Runs a preflight check first and lists the cells to run if anything is missing.


In [11]:
import gradio as gr

preflight(['clip_model', 'image_embeddings', 'images', 'search_images'],
          'L1 - The Finder')

with gr.Blocks(title='L1 - The Finder') as demo_l1:
    gr.Markdown('# L1 - The Finder (visual search)')
    with gr.Row():
        q = gr.Textbox(label='Search query', placeholder="e.g. 'a cat in a hat'")
        k = gr.Slider(1, 10, value=3, step=1, label='Top K')
    btn     = gr.Button('Search', variant='primary')
    gallery = gr.Gallery(label='Results (file name | cosine similarity)')
    btn.click(fn=search_images, inputs=[q, k], outputs=gallery)

demo_l1.launch(debug=True, share=True)


OK  All requirements satisfied for L1 - The Finder.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2cea62fee31f754865.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://2cea62fee31f754865.gradio.live


## Cell 11. Load Gemma 3 VLM (4-bit), *L2 starts here*, **GPU required**

Loads `google/gemma-3-4b-it` in 4-bit via `bitsandbytes`, using `AutoModelForImageTextToText` (the multimodal class: vision encoder + language model). Defines `vlm_generate()`, the single entry point for every Gemma call (captions, L2 answers, L3 multimodal answers). Skip the rest of L2/L3 if you only want the L1 Finder.


In [11]:
import torch
from transformers import AutoProcessor, BitsAndBytesConfig, AutoModelForImageTextToText

if not torch.cuda.is_available():
    print('WARNING  No GPU detected. Runtime -> Change runtime type -> T4 GPU, then re-run from Cell 1.')

quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
vlm_id        = 'google/gemma-3-4b-it'
vlm_processor = AutoProcessor.from_pretrained(vlm_id)
vlm_model     = AutoModelForImageTextToText.from_pretrained(
    vlm_id, quantization_config=quant_config, device_map='auto')
print('OK  Gemma 3 4B loaded in 4-bit.')

def vlm_generate(messages, pil_images=None, max_new_tokens=250):
    """Run Gemma on a chat; return only the newly generated text."""
    text_input = vlm_processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    kwargs = {'text': text_input, 'return_tensors': 'pt'}
    if pil_images:
        kwargs['images'] = pil_images
    inputs = vlm_processor(**kwargs).to(vlm_model.device)
    with torch.no_grad():
        out = vlm_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = out[0][inputs['input_ids'].shape[-1]:]
    return vlm_processor.decode(new_tokens, skip_special_tokens=True).strip()


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.61k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

OK  Gemma 3 4B loaded in 4-bit.


## Cell 12. Image descriptions / captioning (smart cache), *L2*, **GPU required**

Runs Gemma 3 on every image to produce the structured description defined by `CAPTION_PROMPT`. Checkpoints to disk every 10 images; the final JSON is saved to the runtime and mirrored to Drive if mounted. Reads `REGEN['descriptions']` from Cell 5.


In [12]:
import json, torch
from tqdm import tqdm

# Prompt designed so the caption starts with the most discriminative content
# (a keyword line and a short summary) and contains NO preamble or chatty
# opener. This keeps CLIP's first 77 tokens full of distinguishing information.
CAPTION_PROMPT = """You are labelling an image for a high-precision search index.
Do NOT write any preamble, greeting, or sentence like "Okay, here's the analysis".
Start your answer directly with the KEYWORDS line. Use exactly this format:

KEYWORDS: <10-20 comma-separated keywords, ordered from the most visually dominant element to the least. Be specific: name object subtypes, foods by ingredient, materials, colours.>
SUMMARY: <2-3 sentences describing the scene, setting and lighting.>
OBJECTS: <objects you are certain are present, each with a subtype, e.g. "Beverage > Soda > Cola">
PROBABLE: <objects or context that are likely but uncertain>
TEXT: <verbatim transcription of any visible text, or "none">
ATTRIBUTES: <dominant colours, materials, textures>"""

def generate_caption(img):
    messages = [{'role': 'user',
                 'content': [{'type': 'image'}, {'type': 'text', 'text': CAPTION_PROMPT}]}]
    return vlm_generate(messages, pil_images=[img], max_new_tokens=500)

preflight(['vlm_model', 'images'], 'captioning')

if should_regenerate('descriptions'):
    print(f'GENERATING descriptions for {len(images)} images (roughly 20-45 min for 150 on a T4)...')
    descriptions = {}
    for path, img in tqdm(list(zip(image_paths, images)), total=len(images)):
        descriptions[path] = generate_caption(img)
        if len(descriptions) % 10 == 0:                 # checkpoint every 10
            with open(runtime_path('descriptions'), 'w') as f:
                json.dump(descriptions, f)
    with open(runtime_path('descriptions'), 'w') as f:  # final runtime copy
        json.dump(descriptions, f, indent=2)
    save_artifact('descriptions')                       # mirror to Drive (replaces old)
    print(f'OK  {len(descriptions)} descriptions saved.')
else:
    _desc = ensure_runtime_copy('descriptions')
    with open(_desc, 'r') as f:
        descriptions = json.load(f)
    print(f'OK  Loaded {len(descriptions)} cached descriptions.')

_missing = [p for p in image_paths if p not in descriptions]
if _missing:
    print(f'WARNING  {len(_missing)} image(s) have no caption yet. '
          'Set REGEN["descriptions"] = True in Cell 5 and re-run.')
description_texts = [descriptions.get(p, '') for p in image_paths]

OK  All requirements satisfied for captioning.
Using the cached corpus_descriptions.json (set REGEN["descriptions"] = True in Cell 5 to rebuild it).
OK  Loaded 56 cached descriptions.


## Cell 13. Description embeddings (smart cache), *L2*

Encodes every Gemma caption with CLIP's text tower (one vector per caption, same space as the query embeddings). This is the retrieval index for L2 and half of the L3 hybrid score. Reads `REGEN['description_embeddings']` from Cell 5.


In [13]:
import torch, re

def _distinctive_part(caption):
    """Extract the information-dense part of a caption so CLIP's 77-token limit
    is spent on content that distinguishes images, not on shared boilerplate.
    Works with the new KEYWORDS/SUMMARY format AND with older captions that
    still have the '1. DETAILED SUMMARY' headers or an 'Okay, ...' preamble."""
    text = caption

    # 1. New format: the KEYWORDS line (most discriminative, compact).
    m = re.search(r'KEYWORDS?\s*:\s*(.+)', text, flags=re.IGNORECASE)
    kw = m.group(1).split('\n')[0].strip() if m else ''

    # 2. The SUMMARY body (new format) or DETAILED SUMMARY body (old format).
    m = re.search(r'(?:DETAILED\s+)?SUMMARY\s*:?\s*(.+?)(?:\n\s*\*?\*?\d?\.?\s*'
                  r'(?:KEYWORDS?|OBJECTS?|PROBABLE|TEXT|ATTRIBUTES|CONFIRMED)|\Z)',
                  text, flags=re.IGNORECASE | re.DOTALL)
    summary = m.group(1).strip() if m else ''

    # Prefer keywords + summary together; fall back to whichever exists.
    combined = ' '.join(part for part in (kw, summary) if part).strip()
    if combined:
        return combined

    # 3. Last resort: drop a chatty "Okay, ... break down ..." opener.
    return re.sub(r'^\s*Okay[^\n]*\n', '', text, flags=re.IGNORECASE).strip()

def build_description_embeddings(texts):
    """Encode the distinctive part of every caption with CLIP's text tower."""
    return torch.cat([clip_text_features(_distinctive_part(t)) for t in texts])

preflight(['clip_model', 'description_texts'], 'description embeddings')

if should_regenerate('description_embeddings'):
    print('GENERATING description embeddings (using the distinctive part of each caption)...')
    description_embeddings = _ensure_2d(build_description_embeddings(description_texts))
    _save_pickle(runtime_path('description_embeddings'), description_embeddings)
    save_artifact('description_embeddings')
    print(f'OK  Description embeddings ready (shape {tuple(description_embeddings.shape)}).')
else:
    _demb = ensure_runtime_copy('description_embeddings')
    description_embeddings = _ensure_2d(_load_pickle(_demb))
    print(f'OK  Loaded cached description embeddings (shape {tuple(description_embeddings.shape)}).')

OK  All requirements satisfied for description embeddings.
Using the cached description_embeddings.pkl (set REGEN["description_embeddings"] = True in Cell 5 to rebuild it).
OK  Loaded cached description embeddings (shape (56, 512)).


## Cell 14. L2 retrieval & RAG functions

Defines `caption_search_idx` (top-k via the caption embeddings) and `l2_rag_answer` (retrieve by caption similarity, then ask Gemma to answer grounded only in the retrieved captions, returning the supporting images). Instant to run.


In [14]:
import torch.nn.functional as F

def caption_search_idx(query, top_k=3):
    """Top-k image indices + similarity vector via the caption route."""
    q    = embed_query(query)
    sims = F.cosine_similarity(q, description_embeddings)
    top  = sims.topk(min(int(top_k), len(description_embeddings)))
    return top.indices.tolist(), sims

def l2_rag_answer(query, top_k=3):
    """Caption-grounded RAG: retrieve via captions, answer from captions only."""
    idx, sims = caption_search_idx(query, top_k)
    context = '\n\n'.join(f'Image {n+1}: {description_texts[i]}' for n, i in enumerate(idx))
    rag_prompt = (f'Based ONLY on the following image descriptions:\n{context}\n\n'
                  f"Answer the user's question: {query}")
    messages = [{'role': 'user', 'content': [{'type': 'text', 'text': rag_prompt}]}]
    answer  = vlm_generate(messages, max_new_tokens=200)
    gallery = [(images[i], f'sim={sims[i]:.3f}') for i in idx]
    return gallery, answer

print('OK  L2 functions defined.')


OK  L2 functions defined.


## Cell 15. L2: The Companion (Gradio), **GPU required for answers**

Caption-mediated RAG: the query is matched against the Gemma descriptions; Gemma answers using only the retrieved captions, with the retrieved images shown alongside. Preflight first.


In [16]:
import gradio as gr

preflight(['clip_model', 'description_embeddings', 'description_texts',
           'images', 'vlm_model', 'l2_rag_answer'], 'L2 - The Companion')

with gr.Blocks(title='L2 - The Companion') as demo_l2:
    gr.Markdown('# L2 - The Companion (caption-grounded RAG)')
    q       = gr.Textbox(label='Ask a question about your archive')
    k       = gr.Slider(1, 6, value=3, step=1, label='Top K (captions given to the model)')
    btn     = gr.Button('Ask', variant='primary')
    answer  = gr.Textbox(label='VLM answer (grounded in the retrieved captions)')
    gallery = gr.Gallery(label='Retrieved images (evidence)')
    btn.click(fn=l2_rag_answer, inputs=[q, k], outputs=[gallery, answer])

demo_l2.launch(debug=True, share=True)


OK  All requirements satisfied for L2 - The Companion.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://63c7243774662c617a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
[transformers] Deprecated: `processor.image_token` will switch from returning `tokenizer.image_token` to `tokenizer.boi_token` in v5.11.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed 

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://63c7243774662c617a.gradio.live


## Cell 16. Route comparison: L1 (visual) vs L2 (captions)

Runs both retrieval routes on the same queries and prints, per query, the top-k of each route with scores, the overlap, and caption snippets for the text route. Edit `COMPARISON_QUERIES` for your corpus.


In [15]:
import os
import torch.nn.functional as F

COMPARISON_QUERIES = [
    'sushi',
    'a person smiling',
    'red object on a table',
    'handwritten text',
    'night scene with artificial light',
]

def compare_routes(query, top_k=3, snippet_len=160):
    idx_vis, sims_vis = search_images_idx(query, top_k)
    idx_txt, sims_txt = caption_search_idx(query, top_k)
    print('=' * 70)
    print(f'QUERY: "{query}"'); print('-' * 70)
    print('[L1 - visual route: CLIP image embeddings]')
    for i in idx_vis:
        print(f'  {os.path.basename(image_paths[i]):<35} sim={sims_vis[i]:.4f}')
    print('[L2 - caption route: CLIP text embeddings of Gemma captions]')
    for i in idx_txt:
        print(f'  {os.path.basename(image_paths[i]):<35} sim={sims_txt[i]:.4f}')
        print(f'      caption: {description_texts[i][:snippet_len]}...')
    print(f'OVERLAP: {len(set(idx_vis) & set(idx_txt))}/{top_k} images in both routes.')
    print()

preflight(['clip_model', 'image_embeddings', 'description_embeddings',
           'description_texts', 'search_images'], 'route comparison')

for _q in COMPARISON_QUERIES:
    compare_routes(_q)


OK  All requirements satisfied for route comparison.
QUERY: "sushi"
----------------------------------------------------------------------
[L1 - visual route: CLIP image embeddings]
  20210119_135839.jpg                 sim=0.2817
  20210119_135835.jpg                 sim=0.2731
  IMG-20210505-WA0005.jpg             sim=0.2544
[L2 - caption route: CLIP text embeddings of Gemma captions]
  IMG-20210505-WA0005.jpg             sim=0.6354
      caption: KEYWORDS: Japanese food, sushi, dumplings, chopsticks, table, wooden table, orange soda, takeout containers, Asian cuisine, dim sum, Japanese restaurant, wooden...
  20220612_210157.jpg                 sim=0.5922
      caption: KEYWORDS: Korean food, rice, side dishes, bibimbap, chopsticks, beer, table, wooden table, Korean cuisine, small bowls, seafood, vegetables, sauce, Korean meal
...
  IMG_1041.jpg                        sim=0.5734
      caption: KEYWORDS: Takoyaki, Japanese street food, Spherical food, Seafood, Green onion, Sauce, Woo

## Cell 17. L3 hybrid & multimodal functions, *L3 starts here*

Defines `hybrid_search` (min-max-normalised weighted fusion of visual and caption similarity; `alpha` = weight of the visual route) and `l3_multimodal_rag` (passes the retrieved pixels themselves into Gemma's context, capped at `MAX_VLM_IMAGES = 4`). Instant to run.


In [16]:
import torch.nn.functional as F

MAX_VLM_IMAGES = 4   # each image costs ~256 tokens; small VLMs degrade beyond a few

def hybrid_search(query, alpha=0.5, top_k=3):
    """Weighted fusion of the two routes. alpha=1 visual only, alpha=0 captions only."""
    q       = embed_query(query)
    sim_img = F.cosine_similarity(q, image_embeddings)
    sim_txt = F.cosine_similarity(q, description_embeddings)
    sim_img = (sim_img - sim_img.min()) / (sim_img.max() - sim_img.min() + 1e-8)
    sim_txt = (sim_txt - sim_txt.min()) / (sim_txt.max() - sim_txt.min() + 1e-8)
    hybrid  = alpha * sim_img + (1 - alpha) * sim_txt
    top     = hybrid.topk(min(int(top_k), len(image_embeddings)))
    return top.indices.tolist(), hybrid

def l3_multimodal_rag(query, alpha=0.7, top_k=3):
    """Multimodal answering: retrieved images go into Gemma's context (<= MAX_VLM_IMAGES)."""
    top_k = min(int(top_k), MAX_VLM_IMAGES)
    idx, hybrid = hybrid_search(query, alpha, top_k)
    retrieved_images = [images[i] for i in idx]
    content = [{'type': 'text',
                'text': 'Look directly at the provided images to answer the '
                        "user's question. If the images do not contain the answer, "
                        'say so instead of guessing.'}]
    content += [{'type': 'image'} for _ in retrieved_images]
    content.append({'type': 'text', 'text': f'Question: {query}'})
    messages = [{'role': 'user', 'content': content}]
    answer  = vlm_generate(messages, pil_images=retrieved_images, max_new_tokens=250)
    gallery = [(images[i], f'hybrid={hybrid[i]:.3f}') for i in idx]
    return gallery, answer

print('OK  L3 functions defined.')


OK  L3 functions defined.


## Cell 18. L3: The Critic (Gradio), **GPU required**

Hybrid retrieval (the slider blends visual and caption similarity) followed by a multimodal answer: the retrieved images themselves are passed into Gemma's context, capped at 4. Preflight first.


In [17]:
import gradio as gr

preflight(['clip_model', 'image_embeddings', 'description_embeddings',
           'images', 'vlm_model', 'l3_multimodal_rag'], 'L3 - The Critic')

with gr.Blocks(title='L3 - The Critic') as demo_l3:
    gr.Markdown('# L3 - The Critic (hybrid retrieval + multimodal answer)')
    q     = gr.Textbox(label='Ask about your images')
    alpha = gr.Slider(0.0, 1.0, value=0.7, step=0.1,
                      label='Hybrid weight alpha (1 = visual only, 0 = captions only)')
    k     = gr.Slider(1, 4, value=3, step=1, label='Top K (max 4)')
    btn     = gr.Button('Ask (multimodal)', variant='primary')
    answer  = gr.Textbox(label='Multimodal VLM answer')
    gallery = gr.Gallery(label='Images passed to the model (hybrid score)')
    btn.click(fn=l3_multimodal_rag, inputs=[q, alpha, k], outputs=[gallery, answer])

demo_l3.launch(debug=True, share=True)


OK  All requirements satisfied for L3 - The Critic.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3fd375eced6f00ceaa.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
[transformers] Deprecated: `processor.image_token` will switch from returning `tokenizer.image_token` to `tokenizer.boi_token` in v5.11.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://3fd375eced6f00ceaa.gradio.live


## Cell 19. Evaluation: gold-standard queries & precision@k

Measures **precision@k** for the three retrieval routes (CLIP-only, caption-only, hybrid) against a hand-built gold standard.

`GOLD_QUERIES` maps each query to the file names of every image that correctly answers it. It was built by grouping images by keyword (see the optional helper at the end of the notebook) and then **verified by hand against the images**, because keyword presence in a caption does not guarantee the object is the subject of the photo.

The cell validates the gold set first (it warns about file names not in the corpus, and about queries whose few correct images cap their score), then prints a per-query table and the mean per route, and saves `evaluation_results.json` to the runtime (and Drive if mounted). The default run uses **alpha = 0.7**, the value justified by the sweep in the next cell.

`evaluate_all(alpha, save=True)`: set `save=False` to score without overwriting the saved JSON (used by the sweep so intermediate values do not clobber the final file).


In [18]:
import json, os

# ---------------------------------------------------------------------------
# Your gold set. Each query maps to the file names of EVERY image that
# correctly answers it. Verified by hand against the images.
# ---------------------------------------------------------------------------
GOLD_QUERIES = {
    'sushi':      ['20210119_135835.jpg', '20210119_135839.jpg',
                   'IMG-20210505-WA0005.jpg', 'IMG_1043.jpg'],
    'pasta':      ['IMG-20220727-WA0006.jpg'],
    'paella':     ['2008006223-IMG-20220830-WA0014.jpg', '20220824_152600.jpg',
                   '20260221_143735.jpg', '20260607_140135.jpg',
                   '644397870-IMG-20220824-WA0007.jpg', 'IMG-20220830-WA0013.jpg',
                   'IMG-20230305-WA0022.jpg', 'IMG-20231210-WA0000.jpg',
                   'IMG-20251024-WA0003.jpg', 'IMG-20251024-WA0011.jpg'],
    'cat':        ['20210314_102655.jpg', '20210528_174854.jpg'],
    'sandwich':   ['20231206_203332.jpg', 'IMG-20231213-WA0014.jpg'],
    'dessert':    ['20220413_174550.jpg', '20231209_204641.jpg', '20250824_134248.jpg'],
    'soup':       ['20210331_155951.jpg', '20210624_150508.jpg',
                   'IMG-20210331-WA0018.jpg', 'IMG_1045.jpg'],
    'salad':      ['20210624_150507.jpg', '20210624_150508.jpg',
                   '20240213_201342.jpg', '20250821_205503.jpg', 'IMG_1045.jpg'],
    'noodles':    ['20210331_155951.jpg', 'IMG-20210331-WA0018.jpg',
                   'IMG-20210505-WA0005.jpg'],
    'beer':       ['20200806_231754.jpg', '20260216_171158.jpg',
                   'IMG-20231213-WA0014.jpg'],
    'bibimbap':   ['20220612_210157.jpg'],
    'takoyaki':   ['IMG_1041.jpg'],
    'dumplings':  ['20250822_141055.jpg', 'IMG-20210505-WA0005.jpg', 'IMG_1045.jpg'],
    'octopus':    ['IMG_1041.jpg'],
    'ramen':      ['20210331_155951.jpg', 'IMG-20210331-WA0018.jpg'],
    'curry':      ['IMG-20240806-WA0003.jpg'],
    'chicken':    ['20240213_201342.jpg', '20250824_131932.jpg',
                   '20260108_211026.jpg', '20260221_143735.jpg',
                   'IMG-20250623-WA0006.jpg', 'IMG-20260426-WA0002.jpg'],
    'beef':       ['20231208_141413.jpg'],
    'pork':       ['2008006223-IMG-20220830-WA0014.jpg', '20200806_231754.jpg',
                   '20231208_135056.jpg', '20240126_212141.jpg',
                   'IMG-20210505-WA0005.jpg', 'IMG-20231213-WA0014.jpg', 'IMG_1043.jpg'],
    'ice cream':  ['20220413_174550.jpg', '20250824_134248.jpg'],
    'tapas':      ['20241006_112643.jpg', 'IMG-20231213-WA0014.jpg', 'IMG-20251024-WA0011.jpg'],
    'kimchi':     ['20220612_210157.jpg', '20240213_201342.jpg'],
}
K = 3               # evaluate precision at this k
DEFAULT_ALPHA = 0.7 # best fusion weight, justified by the sweep in the next cell

def _names(idx_list):
    return [os.path.basename(image_paths[i]) for i in idx_list]

def precision_at_k(retrieved_names, relevant_names, k):
    """Fraction of the top-k retrieved images that are in the gold set."""
    return len(set(retrieved_names[:k]) & set(relevant_names)) / k

def evaluate_all(alpha=DEFAULT_ALPHA, save=True, verbose=True):
    """Score CLIP-only, caption-only and hybrid retrieval over GOLD_QUERIES.
    * alpha  - fusion weight for the hybrid route.
    * save   - write evaluation_results.json (set False inside the sweep so
               intermediate alphas do not overwrite the final file).
    * verbose- print the per-query table.
    Returns the per-query results dict."""
    preflight(['clip_model', 'image_embeddings', 'description_embeddings',
               'search_images', 'hybrid_search'], 'evaluation')

    known = {os.path.basename(p) for p in image_paths}

    # Validate the gold set before scoring.
    valid = {}
    for query, relevant in GOLD_QUERIES.items():
        unknown = [r for r in relevant if r not in known]
        if unknown and verbose:
            print(f'WARNING  "{query}": gold file(s) not in corpus, skipped: {unknown}')
        good = [r for r in relevant if r in known]
        if good:
            valid[query] = good
            if len(good) < K and verbose:
                print(f'NOTE     "{query}": only {len(good)} correct image(s) exist, '
                      f'so precision@{K} cannot exceed {len(good)/K:.2f} for it.')

    if not valid:
        print('\nNo valid queries. Fill GOLD_QUERIES with real file names.')
        return {}

    results = {}
    if verbose:
        print(f'\n{"query":<42}{"CLIP":>7}{"caption":>9}{"hybrid":>8}')
        print('-' * 66)
    for query, relevant in valid.items():
        p_clip   = precision_at_k(_names(search_images_idx(query, K)[0]),  relevant, K)
        p_capt   = precision_at_k(_names(caption_search_idx(query, K)[0]), relevant, K)
        p_hybrid = precision_at_k(_names(hybrid_search(query, alpha, K)[0]), relevant, K)
        results[query] = {'clip': p_clip, 'caption': p_capt, 'hybrid': p_hybrid}
        if verbose:
            print(f'{query[:40]:<42}{p_clip:>7.2f}{p_capt:>9.2f}{p_hybrid:>8.2f}')

    n = len(results)
    means = {r: sum(v[r] for v in results.values()) / n for r in ('clip', 'caption', 'hybrid')}
    if verbose:
        print('-' * 66)
        print(f'{"MEAN precision@" + str(K):<42}'
              f'{means["clip"]:>7.2f}{means["caption"]:>9.2f}{means["hybrid"]:>8.2f}')
        print(f'\n(scored over {n} valid quer{"y" if n==1 else "ies"}, alpha={alpha})')

    if save:
        out = {'k': K, 'alpha': alpha, 'per_query': results, 'mean': means}
        out_path = os.path.join(RUNTIME_DIR, 'evaluation_results.json')
        with open(out_path, 'w') as f:
            json.dump(out, f, indent=2)
        if DRIVE_DIR:
            import shutil; shutil.copy2(out_path, os.path.join(DRIVE_DIR, 'evaluation_results.json'))
            print('SAVED  evaluation_results.json (alpha=%s) -> runtime AND Google Drive.' % alpha)
        else:
            print('SAVED  evaluation_results.json (alpha=%s) -> runtime ONLY. Download it.' % alpha)
    return results

# Main evaluation at the chosen alpha; this is the file that ships with the project.
_ = evaluate_all(alpha=DEFAULT_ALPHA, save=True)


OK  All requirements satisfied for evaluation.
NOTE     "pasta": only 1 correct image(s) exist, so precision@3 cannot exceed 0.33 for it.
NOTE     "cat": only 2 correct image(s) exist, so precision@3 cannot exceed 0.67 for it.
NOTE     "sandwich": only 2 correct image(s) exist, so precision@3 cannot exceed 0.67 for it.
NOTE     "bibimbap": only 1 correct image(s) exist, so precision@3 cannot exceed 0.33 for it.
NOTE     "takoyaki": only 1 correct image(s) exist, so precision@3 cannot exceed 0.33 for it.
NOTE     "octopus": only 1 correct image(s) exist, so precision@3 cannot exceed 0.33 for it.
NOTE     "ramen": only 2 correct image(s) exist, so precision@3 cannot exceed 0.67 for it.
NOTE     "curry": only 1 correct image(s) exist, so precision@3 cannot exceed 0.33 for it.
NOTE     "beef": only 1 correct image(s) exist, so precision@3 cannot exceed 0.33 for it.
NOTE     "ice cream": only 2 correct image(s) exist, so precision@3 cannot exceed 0.67 for it.
NOTE     "kimchi": only 2 corre

## Cell 20. Fusion weight sweep (justifies alpha = 0.7)

Sweeps the hybrid weight `alpha` from 0 (captions only) to 1 (visual only) and reports the mean precision@k of each setting. This is the empirical justification for the default weight used everywhere else.

Each iteration calls `evaluate_all(..., save=False)` so the sweep does **not** overwrite the shipped `evaluation_results.json`. After the sweep, the cell re-runs the best alpha once with `save=True`, so the saved file always corresponds to the reported optimum.


In [19]:
# Sweep alpha to justify the chosen fusion weight.
# alpha = 1.0 -> visual only; alpha = 0.0 -> captions only.
print(f'{"alpha":>6}{"CLIP":>8}{"caption":>9}{"hybrid":>8}')
print('-' * 31)
best_alpha, best_score = None, -1.0
for a in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    res = evaluate_all(alpha=a, save=False, verbose=False)   # do NOT overwrite the file
    n = len(res)
    m = {r: sum(v[r] for v in res.values()) / n for r in ('clip', 'caption', 'hybrid')}
    print(f'{a:>6.1f}{m["clip"]:>8.2f}{m["caption"]:>9.2f}{m["hybrid"]:>8.2f}')
    if m['hybrid'] > best_score:
        best_score, best_alpha = m['hybrid'], a
print('-' * 31)
print(f'Best hybrid mean precision@{K} = {best_score:.2f} at alpha = {best_alpha}')

# Re-save the results file at the best alpha, so the shipped JSON matches the optimum.
print()
_ = evaluate_all(alpha=best_alpha, save=True, verbose=False)
print(f'evaluation_results.json now reflects the best alpha ({best_alpha}).')


 alpha    CLIP  caption  hybrid
-------------------------------
OK  All requirements satisfied for evaluation.
   0.0    0.38     0.30    0.30
OK  All requirements satisfied for evaluation.
   0.1    0.38     0.30    0.32
OK  All requirements satisfied for evaluation.
   0.2    0.38     0.30    0.36
OK  All requirements satisfied for evaluation.
   0.3    0.38     0.30    0.39
OK  All requirements satisfied for evaluation.
   0.4    0.38     0.30    0.41
OK  All requirements satisfied for evaluation.
   0.5    0.38     0.30    0.41
OK  All requirements satisfied for evaluation.
   0.6    0.38     0.30    0.42
OK  All requirements satisfied for evaluation.
   0.7    0.38     0.30    0.45
OK  All requirements satisfied for evaluation.
   0.8    0.38     0.30    0.44
OK  All requirements satisfied for evaluation.
   0.9    0.38     0.30    0.39
OK  All requirements satisfied for evaluation.
   1.0    0.38     0.30    0.38
-------------------------------
Best hybrid mean precision@3 = 0.45

## Cell 21. Unified Interface (all three levels)

One Gradio app with a tab per level. After the shared setup and each level's data cells are loaded, this is the only interface you need. Preflight covers all three levels.


In [23]:
import gradio as gr

preflight(['clip_model', 'image_embeddings', 'description_embeddings',
           'description_texts', 'images', 'vlm_model',
           'search_images', 'l2_rag_answer', 'l3_multimodal_rag'], 'Unified Interface')

with gr.Blocks(title='Seeing Machines - Unified Interface') as unified_demo:
    gr.Markdown('# Seeing Machines - Unified Interface')
    with gr.Tabs():
        with gr.TabItem('L1 - Finder (visual)'):
            q1 = gr.Textbox(label='Search query')
            k1 = gr.Slider(1, 10, value=3, step=1, label='Top K')
            b1 = gr.Button('Search', variant='primary')
            g1 = gr.Gallery(label='Visual results (file | similarity)')
            b1.click(fn=search_images, inputs=[q1, k1], outputs=g1)
        with gr.TabItem('L2 - Companion (caption RAG)'):
            q2 = gr.Textbox(label='Ask a question (retrieval over captions)')
            k2 = gr.Slider(1, 6, value=3, step=1, label='Top K')
            b2 = gr.Button('Ask', variant='primary')
            a2 = gr.Textbox(label='VLM answer (grounded in captions)')
            g2 = gr.Gallery(label='Supporting images')
            b2.click(fn=l2_rag_answer, inputs=[q2, k2], outputs=[g2, a2])
        with gr.TabItem('L3 - Critic (hybrid multimodal)'):
            q3     = gr.Textbox(label='Ask (retrieved pixels go into the VLM)')
            alpha3 = gr.Slider(0.0, 1.0, value=0.7, step=0.1,
                               label='Hybrid weight alpha (1 = visual, 0 = captions)')
            k3     = gr.Slider(1, 4, value=3, step=1, label='Top K (max 4)')
            b3     = gr.Button('Hybrid search & multimodal answer', variant='primary')
            a3     = gr.Textbox(label='Multimodal answer')
            g3     = gr.Gallery(label='Images passed to the model')
            b3.click(fn=l3_multimodal_rag, inputs=[q3, alpha3, k3], outputs=[g3, a3])

unified_demo.launch(debug=True, share=True)


OK  All requirements satisfied for Unified Interface.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://98f436be374e1a0a91.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/pyth

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://98f436be374e1a0a91.gradio.live


## Cell 22. Experiment: CLIP vs SigLIP2 (model comparison)

An optional add-on experiment that does not affect the main pipeline. It repeats the retrieval evaluation with a second, more recent embedding model, `google/siglip2-base-patch16-224` (Google, 2025), and compares it against the original CLIP (OpenAI, 2021) on the same gold set.

A note on the motivation: SigLIP2's text encoder has a 64 token context, slightly smaller than CLIP's 77, so this does not test "more context fixes subject dilution". Instead it tests a cleaner question: at a comparable token budget, does a newer, better trained encoder retrieve more accurately? The finding that the newer model does not remove the token bottleneck is itself informative, since that limit is structural across this family of models.

The experiment is self contained: it loads SigLIP2, builds its own cached embeddings (separate files, so the CLIP artefacts are untouched), scores precision@k over the existing `GOLD_QUERIES`, and produces a comparison chart saved as `model_comparison.png` for the documentation.

Requires the CLIP evaluation (Cell 19) to have run, since it reuses `GOLD_QUERIES`, `image_paths`, `descriptions` and the metric helpers.


In [20]:
import os, pickle, json
import torch, torch.nn.functional as F
from PIL import Image
from transformers import AutoModel, AutoProcessor

# --- Guard: we need the corpus, captions and gold set already in memory -------
preflight(['image_paths', 'descriptions', 'search_images_idx'], 'CLIP vs SigLIP2 experiment')
if 'GOLD_QUERIES' not in globals():
    raise RuntimeError('Run Cell 19 first: GOLD_QUERIES and the metric helpers must exist.')

# --- Cache paths for the SigLIP2 artefacts (separate from CLIP) ---------------
SIGLIP_IMG_CACHE = os.path.join(RUNTIME_DIR, 'siglip2_image_embeddings.pkl')
SIGLIP_TXT_CACHE = os.path.join(RUNTIME_DIR, 'siglip2_desc_embeddings.pkl')

def _sig_save(path, obj):
    with open(path, 'wb') as f: pickle.dump(obj, f)
    if DRIVE_DIR:
        import shutil; shutil.copy2(path, os.path.join(DRIVE_DIR, os.path.basename(path)))

def _sig_load(path):
    rp = path
    if not os.path.exists(rp) and DRIVE_DIR:
        dp = os.path.join(DRIVE_DIR, os.path.basename(path))
        if os.path.exists(dp):
            import shutil; shutil.copy2(dp, rp)
    return pickle.load(open(rp, 'rb')) if os.path.exists(rp) else None

# --- Load SigLIP2 (base variant fits comfortably on a T4) --------------------
SIGLIP_ID = 'google/siglip2-base-patch16-224'
print(f'Loading {SIGLIP_ID} ...')
sig_model     = AutoModel.from_pretrained(SIGLIP_ID).to(device)
sig_processor = AutoProcessor.from_pretrained(SIGLIP_ID)
sig_model.eval()

def _sig_as_tensor(out):
    """Return a plain tensor from either a tensor or a model output object."""
    if isinstance(out, torch.Tensor):
        return out
    for attr in ('image_embeds', 'text_embeds', 'pooler_output'):
        v = getattr(out, attr, None)
        if v is not None:
            return v
    raise TypeError(f'Unexpected SigLIP output: {type(out)}')

def sig_image_features(pil_image):
    inputs = sig_processor(images=pil_image, return_tensors='pt').to(device)
    with torch.no_grad():
        return _sig_as_tensor(sig_model.get_image_features(**inputs)).cpu()

def sig_text_features(text):
    # SigLIP2 requires padding='max_length', max_length=64 for correct embeddings.
    inputs = sig_processor(text=[text], return_tensors='pt',
                           padding='max_length', max_length=64, truncation=True).to(device)
    with torch.no_grad():
        return _sig_as_tensor(sig_model.get_text_features(**inputs)).cpu()

# --- Build (or load) SigLIP2 embeddings for images and captions --------------
sig_img_emb = _sig_load(SIGLIP_IMG_CACHE)
if sig_img_emb is None:
    print('Embedding images with SigLIP2 ...')
    sig_img_emb = torch.cat([sig_image_features(Image.open(p).convert('RGB')) for p in image_paths])
    _sig_save(SIGLIP_IMG_CACHE, sig_img_emb)
else:
    print('Loaded cached SigLIP2 image embeddings.')

sig_txt_emb = _sig_load(SIGLIP_TXT_CACHE)
if sig_txt_emb is None:
    print('Embedding captions with SigLIP2 ...')
    # Reuse the same "distinctive part" of each caption used for the CLIP route.
    texts = [_distinctive_part(descriptions.get(p, '')) for p in image_paths]
    sig_txt_emb = torch.cat([sig_text_features(t) for t in texts])
    _sig_save(SIGLIP_TXT_CACHE, sig_txt_emb)
else:
    print('Loaded cached SigLIP2 caption embeddings.')

# --- SigLIP2 retrieval (mirrors the CLIP search functions) -------------------
def sig_query(q):
    return sig_text_features(q)

def sig_visual_idx(query, top_k=3):
    sims = F.cosine_similarity(sig_query(query), sig_img_emb)
    return sims.topk(min(top_k, len(sig_img_emb))).indices.tolist()

def sig_caption_idx(query, top_k=3):
    sims = F.cosine_similarity(sig_query(query), sig_txt_emb)
    return sims.topk(min(top_k, len(sig_txt_emb))).indices.tolist()

# --- Score both models on the same gold set ----------------------------------
known = {os.path.basename(p) for p in image_paths}
valid = {q: [r for r in rel if r in known]
         for q, rel in GOLD_QUERIES.items()
         if any(r in known for r in rel)}

def _p(idxs, relevant):
    names = [os.path.basename(image_paths[i]) for i in idxs]
    return len(set(names[:K]) & set(relevant)) / K

rows = {}
for q, rel in valid.items():
    rows[q] = {
        'clip_visual'   : _p(search_images_idx(q, K)[0], rel),
        'siglip_visual' : _p(sig_visual_idx(q, K), rel),
        'clip_caption'  : _p(caption_search_idx(q, K)[0], rel),
        'siglip_caption': _p(sig_caption_idx(q, K), rel),
    }

n = len(rows)
means = {k: sum(r[k] for r in rows.values()) / n for k in
         ('clip_visual', 'siglip_visual', 'clip_caption', 'siglip_caption')}

print(f'\nMean precision@{K} over {n} gold queries')
print('-' * 46)
print(f'{"":16}{"visual":>12}{"caption":>14}')
print(f'{"CLIP (2021)":16}{means["clip_visual"]:>12.3f}{means["clip_caption"]:>14.3f}')
print(f'{"SigLIP2 (2025)":16}{means["siglip_visual"]:>12.3f}{means["siglip_caption"]:>14.3f}')
print('-' * 46)

# Save the numbers alongside the chart.
comp = {'k': K, 'n_queries': n, 'means': means, 'per_query': rows}
with open(os.path.join(RUNTIME_DIR, 'model_comparison.json'), 'w') as f:
    json.dump(comp, f, indent=2)
if DRIVE_DIR:
    import shutil; shutil.copy2(os.path.join(RUNTIME_DIR, 'model_comparison.json'),
                               os.path.join(DRIVE_DIR, 'model_comparison.json'))


OK  All requirements satisfied for CLIP vs SigLIP2 experiment.
Loading google/siglip2-base-patch16-224 ...


config.json:   0%|          | 0.00/253 [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.2k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/34.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Loaded cached SigLIP2 image embeddings.
Loaded cached SigLIP2 caption embeddings.

Mean precision@3 over 22 gold queries
----------------------------------------------
                      visual       caption
CLIP (2021)            0.379         0.303
SigLIP2 (2025)         0.576         0.197
----------------------------------------------


## Cell 23. Comparison chart (for the documentation)

Renders the CLIP vs SigLIP2 result as a grouped bar chart and saves it as `model_comparison.png` (runtime and Drive). Drop this image into the documentation site.


In [21]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

labels = ['Visual route', 'Caption route']
clip_vals   = [means['clip_visual'],   means['clip_caption']]
siglip_vals = [means['siglip_visual'], means['siglip_caption']]

x = range(len(labels))
w = 0.36
fig, ax = plt.subplots(figsize=(7, 4.4), dpi=150)
b1 = ax.bar([i - w/2 for i in x], clip_vals,   w, label='CLIP (2021)',    color='#8f8f8f')
b2 = ax.bar([i + w/2 for i in x], siglip_vals, w, label='SigLIP2 (2025)', color='#3a4a7a')

ax.set_ylabel(f'Mean precision@{K}')
ax.set_title(f'Retrieval accuracy by model and route (n={n} queries)')
ax.set_xticks(list(x)); ax.set_xticklabels(labels)
ax.set_ylim(0, max(clip_vals + siglip_vals) * 1.25 + 0.01)
ax.legend(frameon=False)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
for bars in (b1, b2):
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f'{h:.2f}', (bar.get_x() + bar.get_width()/2, h),
                    textcoords='offset points', xytext=(0, 3),
                    ha='center', fontsize=9, color='#1a1a1a')
plt.tight_layout()

out = os.path.join(RUNTIME_DIR, 'model_comparison.png')
plt.savefig(out, bbox_inches='tight')
if DRIVE_DIR:
    import shutil; shutil.copy2(out, os.path.join(DRIVE_DIR, 'model_comparison.png'))
    print('SAVED  model_comparison.png -> runtime AND Google Drive.')
else:
    print('SAVED  model_comparison.png -> runtime ONLY. Download it for the docs.')
plt.show()


SAVED  model_comparison.png -> runtime ONLY. Download it for the docs.


## Cell 24. Optional utility: group images by concept (gold-set builder)

Not part of the pipeline. This helper lists, for each concept, the images whose captions mention it, which is how the gold set was drafted before being verified by hand. Edit `concepts` and run it whenever you want to extend or rebuild `GOLD_QUERIES`.


In [22]:
# TEMPORARY HELPER - group images by concept using the captions Gemma produced.
# Edit `concepts`, run the cell, copy correct file names into GOLD_QUERIES (Cell 19).
concepts = [
    'dumplings', 'shrimp', 'octopus', 'tempura', 'ramen', 'curry',
    'chicken', 'beef', 'pork', 'egg', 'bread', 'cheese', 'wine',
    'vegetables', 'cake', 'ice cream', 'tapas', 'kimchi',
]

for concept in concepts:
    hits = [image_paths[i].split('/')[-1]
            for i, p in enumerate(image_paths)
            if concept.lower() in descriptions.get(p, '').lower()]
    print(f'{concept:<12} ({len(hits)}): {hits}')

dumplings    (3): ['20250822_141055.jpg', 'IMG-20210505-WA0005.jpg', 'IMG_1045.jpg']
shrimp       (10): ['2008006223-IMG-20220830-WA0014.jpg', '20220824_152600.jpg', '20260221_143735.jpg', '20260607_140135.jpg', '644397870-IMG-20220824-WA0007.jpg', 'IMG-20220830-WA0013.jpg', 'IMG-20230305-WA0022.jpg', 'IMG-20231210-WA0000.jpg', 'IMG-20251024-WA0003.jpg', 'IMG-20251024-WA0011.jpg']
octopus      (3): ['20240213_201346.jpg', 'IMG-20230305-WA0022.jpg', 'IMG_1041.jpg']
tempura      (0): []
ramen        (2): ['20210331_155951.jpg', 'IMG-20210331-WA0018.jpg']
curry        (1): ['IMG-20240806-WA0003.jpg']
chicken      (7): ['20240213_201342.jpg', '20240213_201346.jpg', '20250824_131932.jpg', '20260108_211026.jpg', '20260221_143735.jpg', 'IMG-20250623-WA0006.jpg', 'IMG-20260426-WA0002.jpg']
beef         (1): ['20231208_141413.jpg']
pork         (7): ['2008006223-IMG-20220830-WA0014.jpg', '20200806_231754.jpg', '20231208_135056.jpg', '20240126_212141.jpg', 'IMG-20210505-WA0005.jpg', 'IMG-2023121